# Hybrid Data Ingestion for Oracle 23ai (HR Recruitment)

**Copyright 2026, Denis Rothman**

This notebook implements a **Hybrid RAG** pipeline, demonstrating the true power of the "Converged Database." Unlike the previous scenario which focused solely on unstructured text files, this notebook ingests **Structured Data** (SQL columns) alongside **Unstructured Data** (Vector Embeddings).

We assume the role of the **Data Engineer** building a **Smart Recruitment Platform**. We will populate the database with candidate profiles that contain both hard constraints (Salary, Years of Experience) and soft semantic data (Resume Summaries).

### Notebook Highlights:

* **Use Case:** A **"Smart Recruitment Platform"**. We simulate a dataset of job candidates to demonstrate how AI can filter by hard constraints (e.g., *"Must have 5+ years experience"*) while simultaneously searching for semantic fit (e.g., *"Has leadership potential"*).
* **Infrastructure:** Establishes secure connectivity using Oracle Wallets and the `oracledb` thin client.
* **Data Pipeline:**
    * **Data Staging:** Dynamic generation of structured candidate profiles (JSON/Dict).
    * **Hybrid Transformation:** Selective vectorization. We embed the *Resume Summary* while keeping *Salary* and *Experience* as standard SQL integers.
    * **Loading:** efficient batch insertion into the hybrid `CANDIDATE_POOL` table and the `RECRUITMENT_RULES` context table.
* **Validation:** Includes a **Hybrid Query** test that filters results using both SQL `WHERE` clauses and Vector `ORDER BY` distance.

## ⚠️ Instructions: HR Data Ingestion Workflow

**Crucial Step:** This notebook populates the **HR Recruitment Schema** (Hybrid Data). To ensure the "Vector Enrichment" process works correctly and to avoid data inconsistencies, please follow this workflow:

1.  **Prerequisites:**
    * Ensure you have successfully ran the **`1_DBA_Oracle_Management_V2.ipynb`** notebook with `create_tables = True`.
    * Verify that the `CANDIDATE_POOL` and `RECRUITMENT_RULES` tables exist in your database.

2.  **Run Frequency:**
    * **Run this notebook ONLY ONCE per session.**
    * While this notebook uses `MERGE` to handle duplicates, running it multiple times can confuse the vectorization steps (which look for empty vectors) and make it harder to verify your results.

3.  **Troubleshooting (Clean Slate Protocol):**
    * If you need to restart the ingestion process (e.g., you changed the source data or want to see the process from scratch), you must **clean the database first**.
    * **Action:** Go back to **`1_DBA_Oracle_Management_V2.ipynb`** (or your `Oracle_DBA_Console`).
    * **Config:** Set `empty_tables = True`.
    * **Execute:** Run the notebook to truncate (wipe) the `CANDIDATE_POOL` and `RECRUITMENT_RULES` tables.
    * **Retry:** Return here and run this notebook again.

### ℹ️ Note: Shared Schema Management

Please be aware that this notebook (**Version 2**) manages the **entire** vector schema for the course, encompassing both the General RAG (Chapter 2) and Hybrid RAG (Chapter 3) tables.

**Managed Tables:**

1.  **Chapter 2 (General RAG):**
    * `CONTEXT_LIBRARY` (Blueprints)
    * `KNOWLEDGE_STORE` (Unstructured Documents)

2.  **Chapter 3 (HR Recruitment - New):**
    * `CANDIDATE_POOL` (Hybrid: SQL + Vector)
    * `RECRUITMENT_RULES` (Agent Personas)

**⚠️ Warning:** If you set `empty_tables = True` in this notebook, it will truncate **ALL** four tables listed above.

# 1.Installation and Setup

## 1.3.Installing OpenAI

Run this section first to avoid dependency conflicts with other installations.

In [1]:
#!pip install tqdm==4.67.1 --upgrade
!pip install openai==2.14.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 9.3 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.24.0
    Uninstalling openai-2.24.0:
      Successfully uninstalled openai-2.24.0


In [2]:
# Imports and API Key Setup
# We will use the OpenAI library to interact with the LLM and Google Colab's
# secret manager to securely access your API key.

import os
from openai import OpenAI
from google.colab import userdata

# Load the API key from Colab secrets, set the env var, then init the client
try:
    api_key = userdata.get("API_KEY")
    if not api_key:
        raise userdata.SecretNotFoundError("API_KEY not found.")

    # Set environment variable for downstream tools/libraries
    os.environ["OPENAI_API_KEY"] = api_key

    # Create client (will read from OPENAI_API_KEY)
    client = OpenAI()
    print("OpenAI API key loaded and environment variable set successfully.")

except userdata.SecretNotFoundError:
    print('Secret "API_KEY" not found.')
    print('Please add your OpenAI API key to the Colab Secrets Manager.')
except Exception as e:
    print(f"An error occurred while loading the API key: {e}")

# Configuration
EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM = 1536 # Dimension for text-embedding-3-small
GENERATION_MODEL = "gpt-5.2"

OpenAI API key loaded and environment variable set successfully.


In [3]:
#@title Retrieving GitHub Private token (this cell will be deleted when the repository is public)
import os
from openai import OpenAI
from google.colab import userdata

# Load the API key from Colab secrets, set the env var, then init the client
try:
    pt = userdata.get("GITHUB_TOKEN")
    pt=pt.strip()
    if not pt:
        raise userdata.SecretNotFoundError("GITHUB_TOKEN not found.")

    print("GITHUB_TOKENAPI private token loaded successfully.")

except userdata.SecretNotFoundError:
    print('Secret "GITHUB_TOKEN" not found.')
    print('Please activate or add your GITHUB_TOKEN private token to the Colab Secrets Manager.')
except Exception as e:
    print(f"An error occurred while loading the GITHUB_TOKEN: {e}")

GITHUB_TOKENAPI private token loaded successfully.


## DbA Parameters


## 1.1 Oracle environment Setup & Wallet Management

This step prepares the runtime environment by connecting to Google Drive and ensuring the Oracle Wallet is available. The Oracle Wallet contains the SSL/TLS certificates and configuration files (tnsnames.ora, sqlnet.ora) necessary for a secure connection to the Oracle Autonomous Database.

Google Drive Mount: Maps your personal Drive to /content/drive, allowing the notebook to read credentials and configuration files stored externally.

Wallet Unzipping: If unzip_wallet is set to True, the script searches for the wallet ZIP file in the specified path and extracts it. This ensures the connection drivers can locate the required security certificates.

Path Definition: Sets wallet_path to the directory where the unzipped configuration files reside, which will be passed to the Oracle driver in subsequent steps.

In [4]:
from google.colab import drive
drive.mount('/content/drive')
wallet_path = "/content/drive/MyDrive/oracle_wallet"

Mounted at /content/drive


## 1.2 Install Oracle Database Driver
This command installs the oracledb Python driver, which is the renamed and modernized successor to the legacy cx_Oracle driver. It serves as the bridge between this Python notebook and the Oracle Database.

Version Pinning (==3.4.1): The version is explicitly fixed to 3.4.1 to ensure stability and reproducibility. In production DBA scripts, pinning versions prevents unexpected updates or API changes from breaking the automation workflow.

Thin Client Mode: By default, this driver operates in "Thin" mode, meaning it communicates directly with the database using pure Python code without requiring local Oracle Client libraries (Instant Client).

In [5]:
!pip install oracledb==3.4.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 14.6 MB/s eta 0:00:00


In [6]:
import oracledb
print(oracledb.__version__)

3.4.1


### Establish Secure Database Connection
This step handles the authentication and connection to the Oracle 23ai instance. It is designed to adhere to security best practices by strictly separating code from credentials.

External Credential Management: Instead of hardcoding sensitive passwords directly into the notebook (which is a security risk), the script reads a credentials.txt file stored securely on Google Drive.

Credential Parsing: The read_key_value_file helper function parses the external file to retrieve the username, password, Wallet password, and DSN (Data Source Name).

Connection Initialization: The oracledb.connect() method establishes the session using the extracted credentials and the local Wallet configuration path.

Connectivity Test: A simple "Heartbeat" query (SELECT ... FROM DUAL) is executed immediately to verify that the connection is active and stable before proceeding.

In [7]:
import os
from google.colab import userdata
oracle_dsn = userdata.get('oracle_dsn')

creds_path = "/content/drive/MyDrive/files/oracle/credentials.txt"
if not os.path.exists(creds_path):
    raise FileNotFoundError(f"Credentials file not found: {creds_path}")

def read_key_value_file(path):
    creds = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if "=" not in line:
                continue
            k, v = line.split("=", 1)
            creds[k.strip()] = v.strip()
    return creds

creds = read_key_value_file(creds_path)

# Use values (do not print them)
user = creds.get("user")
password = creds.get("password")
wallet_password = creds.get("wallet_password")
dsn = creds.get("dsn", oracle_dsn)
wallet_path = creds.get("wallet_path", "/content/drive/MyDrive/files/oracle")

import oracledb
connection = oracledb.connect(
    user=user,
    password=password,
    dsn=dsn,
    config_dir=wallet_path,
    wallet_location=wallet_path,
    wallet_password=wallet_password
)

cursor = connection.cursor()
cursor.execute("SELECT 'Connected!' FROM dual")
print(cursor.fetchone())

('Connected!',)


## 1.4. Additional imports

In [8]:
# Imports for this notebook
import json
import time
from tqdm.auto import tqdm
import tiktoken                                                         # tokenization
from tenacity import retry, stop_after_attempt, wait_random_exponential # to retry functions
import re
import textwrap
from IPython.display import display, Markdown
import copy

# 2.Data Preparation: HR Candidates & Rules

### 2.1 Initialize Staging Directory

We create a local directory to store the incoming HR dataset (`hr_data.json`) before processing it.


## 2.2 Data Acquisition



In [9]:
retrieve_evidence = True #@param {type:"boolean"}

In [10]:
if retrieve_evidence==True:
  !curl -L -H "Authorization: token {pt}" -H "Accept: application/vnd.github.v3.raw" "https://api.github.com/repos/Denis2054/RAG-Driven-Generative-AI-2nd-Edition/contents/enterprise_data/talent_acquisition/hr_data.json" -o hr_data.json

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  3612  100  3612    0     0  10514      0 --:--:-- --:--:-- --:--:-- 10500


In [11]:
# 4. Load HR Data into Memory
# -------------------------------------------------------------------------
import json
import os

# Initialize storage
hr_data = {}
file_path = "hr_data.json" # Matching the -o from your curl command

# Validation Gate
if os.path.exists(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            hr_data = json.load(f)

        print(f"✅ HR Data Loaded Successfully")
        print(f"   - Candidates found: {len(hr_data.get('candidates', []))}")
        print(f"   - Recruitment Rules found: {len(hr_data.get('rules', []))}")

    except json.JSONDecodeError as e:
        print(f"❌ Error decoding JSON: {e}")
else:
    print(f"❌ Error: '{file_path}' not found. Did you run the curl command?")

✅ HR Data Loaded Successfully
   - Candidates found: 5
   - Recruitment Rules found: 3


# 3.Helper Functions for Chunking and Embedding

Before we can upload data to Oracle, we need two critical utilities:

chunk_text: A function to break large documents (like the "Patch Notes") into smaller, manageable pieces that fit within the AI's context window. We use tiktoken to ensure we split by tokens, not just characters.

get_embeddings_batch: A function that sends text to OpenAI's API and returns the vector representation (VECTOR(1536)). This is the bridge between human language and the Oracle Vector database.

In [12]:
# 4. Helper Functions for Chunking and Embedding
# -------------------------------------------------------------------------

# Initialize tokenizer for robust, token-aware chunking
tokenizer = tiktoken.get_encoding("cl100k_base")

def chunk_text(text, chunk_size=400, overlap=50):
    """
    Chunks text based on token count with overlap.
    Best practice for RAG: Overlap ensures context isn't lost at the cut.
    """
    tokens = tokenizer.encode(text)
    chunks = []
    for i in range(0, len(tokens), chunk_size - overlap):
        chunk_tokens = tokens[i:i + chunk_size]
        chunk_text = tokenizer.decode(chunk_tokens)
        # Basic cleanup to remove excessive newlines
        chunk_text = chunk_text.replace("\n", " ").strip()
        if chunk_text:
            chunks.append(chunk_text)
    return chunks

@retry(wait=wait_random_exponential(min=1, max=60), stop=stop_after_attempt(6))
def get_embeddings_batch(texts, model=EMBEDDING_MODEL):
    """
    Generates embeddings for a batch of texts using OpenAI, with exponential backoff retries.
    """
    # OpenAI recommends replacing newlines with spaces for best embedding results
    texts = [t.replace("\n", " ") for t in texts]

    # Call the OpenAI API
    response = client.embeddings.create(input=texts, model=model)

    # Return just the list of embedding vectors
    return [item.embedding for item in response.data]

# 4.Processing and uploading the data to Oracle

In [13]:
#@title 4.1 Populate Tables (Structured Data Only)
# -------------------------------------------------------------------------
# We insert the raw relational data first. The 'vectors' will be generated later.

populate_tables = True #@param {type:"boolean"}

if populate_tables:
    print("--- Populating SQL Tables (Structured Data) ---\n")
    cursor = connection.cursor()

    # 1. Insert Candidates (Using Dictionary Binding to handle the ID merge)
    candidates = hr_data.get("candidates", [])
    for cand in candidates:
        cursor.execute("""
            MERGE INTO candidate_pool target
            USING (SELECT :candidate_id AS id FROM dual) source
            ON (target.candidate_id = source.id)
            WHEN NOT MATCHED THEN
                INSERT (candidate_id, full_name, years_experience, salary_expectation, skills, summary)
                VALUES (:candidate_id, :full_name, :years_experience, :salary_expectation, :skills, :summary)
        """, {
            "candidate_id": cand['candidate_id'],
            "full_name": cand['full_name'],
            "years_experience": cand['years_experience'],
            "salary_expectation": cand['salary_expectation'],
            "skills": cand['skills'],
            "summary": cand['summary']
        })
    print(f"-> Merged {len(candidates)} rows into CANDIDATE_POOL.")

    # 2. Insert Rules (Using Dictionary Binding)
    rules = hr_data.get("rules", [])
    for rule in rules:
        cursor.execute("""
            MERGE INTO recruitment_rules target
            USING (SELECT :rule_id AS id FROM dual) source
            ON (target.rule_id = source.id)
            WHEN NOT MATCHED THEN
                INSERT (rule_id, agent_persona, evaluation_criteria)
                VALUES (:rule_id, :agent_persona, :evaluation_criteria)
        """, {
            "rule_id": rule['rule_id'],
            "agent_persona": rule['agent_persona'],
            "evaluation_criteria": rule['evaluation_criteria']
        })
    print(f"-> Merged {len(rules)} rows into RECRUITMENT_RULES.")

    connection.commit()
    print("\n✅ SQL Population Complete.")

--- Populating SQL Tables (Structured Data) ---

-> Merged 5 rows into CANDIDATE_POOL.
-> Merged 3 rows into RECRUITMENT_RULES.

✅ SQL Population Complete.


In [14]:
#@title 4.2 Vectorization from Tables (Enrichment)
# -------------------------------------------------------------------------
# We fetch the rows where we just inserted text, generate embeddings,
# and UPDATE the table with the vectors.

vectorize_db = True #@param {type:"boolean"}

if vectorize_db:
    print("--- Starting Vector Enrichment (Table -> Vector -> Update) ---\n")
    cursor = connection.cursor()

    # --- A. Enrich Candidate Pool ---
    # We fetch ID and SUMMARY (the text to vectorize)
    cursor.execute("SELECT candidate_id, summary FROM candidate_pool WHERE resume_vector IS NULL")
    rows_to_process = cursor.fetchall()

    if rows_to_process:
        print(f"Vectorizing {len(rows_to_process)} candidates...")
        # FIX: Explicitly read the LOB object into a string
        summaries = [row[1].read() for row in rows_to_process]

        vectors = get_embeddings_batch(summaries)

        # Update the table
        for i, (cand_id, _) in enumerate(rows_to_process):
            cursor.setinputsizes(vec=oracledb.DB_TYPE_VECTOR)
            cursor.execute("""
                UPDATE candidate_pool
                SET resume_vector = :vec
                WHERE candidate_id = :id
            """, {"vec": vectors[i], "id": cand_id})

        connection.commit()
        print("-> Candidates updated with vectors.")
    else:
        print("-> No new candidates to vectorize.")


    # --- B. Enrich Recruitment Rules ---
    # We combine Persona + Criteria for the embedding context
    cursor.execute("SELECT rule_id, agent_persona || ' ' || evaluation_criteria FROM recruitment_rules WHERE rule_vector IS NULL")
    rows_to_process = cursor.fetchall()

    if rows_to_process:
        print(f"Vectorizing {len(rows_to_process)} rules...")

        # FIX: Check if the result is a LOB (CLOB) or a String (VARCHAR2) and handle both
        texts = []
        for row in rows_to_process:
            data = row[1]
            if hasattr(data, 'read'):
                texts.append(data.read()) # It's a CLOB
            else:
                texts.append(str(data))   # It's already a string

        vectors = get_embeddings_batch(texts)

        for i, (rule_id, _) in enumerate(rows_to_process):
            cursor.setinputsizes(vec=oracledb.DB_TYPE_VECTOR)
            cursor.execute("""
                UPDATE recruitment_rules
                SET rule_vector = :vec
                WHERE rule_id = :id
            """, {"vec": vectors[i], "id": rule_id})

        connection.commit()
        print("-> Rules updated with vectors.")
    else:
        print("-> No new rules to vectorize.")

    print("\n✅ Vector Enrichment Complete.")

--- Starting Vector Enrichment (Table -> Vector -> Update) ---

Vectorizing 5 candidates...
-> Candidates updated with vectors.
Vectorizing 3 rules...
-> Rules updated with vectors.

✅ Vector Enrichment Complete.


# 5.Final Verification: Hybrid Query (SQL + Vector)

We verify the system by running a **Hybrid Search**. This demonstrates the power of the Converged Database by applying two filters simultaneously:

1.  **Structured Filter (SQL):** We only want candidates with `salary_expectation <= 170000`.
2.  **Semantic Filter (Vector):** We semantically search for candidates with *"Leadership and team building"* skills.

The database engine executes both constraints in a single query execution plan.

In [15]:
import oracledb

# 1. Define the Hybrid Query
# We want a leader, but we have a budget cap.
user_query = "Leadership and team building experience"
max_budget = 170000

print(f"🔎 Query: '{user_query}'")
print(f"💰 Constraint: Salary <= ${max_budget}\n")

# 2. Generate Vector for the text query
query_vector = get_embeddings_batch([user_query])[0]

# 3. Execute Hybrid SQL
# Note: We filter by SQL (salary) AND order by Vector Distance (summary)
cursor.setinputsizes(v=oracledb.DB_TYPE_VECTOR)
cursor.execute("""
    SELECT candidate_id, full_name, salary_expectation, summary,
           VECTOR_DISTANCE(resume_vector, :v, DOT) as similarity
    FROM candidate_pool
    WHERE salary_expectation <= :budget
    ORDER BY similarity DESC
    FETCH FIRST 3 ROWS ONLY
""", {"v": query_vector, "budget": max_budget})

# 4. Display Results
results = cursor.fetchall()

print("--- Hybrid Search Results ---")
for r in results:
    cand_id, name, salary, summary_lob, score = r

    # FIX: Convert LOB to string
    summary_text = summary_lob.read()

    print(f"Candidate: {name} (ID: {cand_id})")
    print(f"Salary: ${salary:,}")
    print(f"Match Score: {score:.4f}")
    print(f"Summary Snippet: {summary_text[:100]}...")
    print("-" * 50)

🔎 Query: 'Leadership and team building experience'
💰 Constraint: Salary <= $170000

--- Hybrid Search Results ---
Candidate: Riley S. (ID: CAND_004)
Salary: $140,000
Match Score: -0.2124
Summary Snippet: Backend Java Developer focused on banking transaction systems. Solid experience with Oracle Database...
--------------------------------------------------
Candidate: Jordan L. (ID: CAND_002)
Salary: $85,000
Match Score: -0.2528
Summary Snippet: Junior Data Scientist recently graduated with a Masters in AI. Strong academic background in machine...
--------------------------------------------------
Candidate: Alex V. (ID: CAND_001)
Salary: $165,000
Match Score: -0.3072
Summary Snippet: Senior Full Stack Engineer with deep expertise in cloud-native architectures. specialized in buildin...
--------------------------------------------------


# 6.System Summary

In [16]:
print("\n=== Oracle Hybrid Table Summary ===\n")

# --- 1. Check HR Tables ---
tables = ['CANDIDATE_POOL', 'RECRUITMENT_RULES']

for t in tables:
    try:
        cursor.execute(f"SELECT COUNT(*) FROM {t}")
        count = cursor.fetchone()[0]
        print(f"Table {t}: {count} rows")
    except Exception as e:
        print(f"Table {t}: Not Found or Error ({e})")

print("\n--- Sample Candidate (Structured + Vector) ---")
cursor.execute("""
    SELECT candidate_id, full_name, salary_expectation,
           CASE WHEN resume_vector IS NOT NULL THEN '✅ Vector Present' ELSE '❌ No Vector' END
    FROM candidate_pool
    FETCH FIRST 2 ROWS ONLY
""")
for row in cursor.fetchall():
    print(row)

print("\n=== Verification Complete ===")


=== Oracle Hybrid Table Summary ===

Table CANDIDATE_POOL: 5 rows
Table RECRUITMENT_RULES: 3 rows

--- Sample Candidate (Structured + Vector) ---
('CAND_005', 'Quinn R.', 155000, '✅ Vector Present')
('CAND_003', 'Casey M.', 210000, '✅ Vector Present')

=== Verification Complete ===
